# @zz-windfm-test/akkudoktor-eos

Akkudoktor EOS: genetic-algorithm optimisation of home battery, PV, EV and appliances. Tool: 24-hour plan.

Published version **v1.0.0** — this notebook fetches that exact snapshot from the
PowerAI Hub, not the upstream repository's latest commit.

Run all cells: **Runtime → Run all** (or ⌘/Ctrl + F9).

The Hub does not run primitives. This notebook runs in *your* Colab session, on
Google's hardware, under your account.

[View on the Hub](https://hub.powerai.ai/zz-windfm-test/akkudoktor-eos)


In [ ]:
PRIMITIVE = "@zz-windfm-test/akkudoktor-eos"
ARCHIVE = "https://hub-api.powerai.ai/api/artifacts/zz-windfm-test/akkudoktor-eos/download-archive/"
WORKDIR = "/content/primitive"

import shutil, urllib.request, zipfile
from pathlib import Path

shutil.rmtree(WORKDIR, ignore_errors=True)          # re-runs start clean
Path(WORKDIR).mkdir(parents=True, exist_ok=True)

archive = Path("/content/primitive.zip")
urllib.request.urlretrieve(ARCHIVE, archive)
with zipfile.ZipFile(archive) as zf:
    zf.extractall(WORKDIR)

files = sorted(p for p in Path(WORKDIR).rglob("*") if p.is_file())
print(f"{PRIMITIVE}: {len(files)} files in {WORKDIR}")
for p in files[:20]:
    print(" ", p.relative_to(WORKDIR))
if len(files) > 20:
    print(f"  … and {len(files) - 20} more")


In [ ]:
import importlib.metadata as md, os, subprocess, sys
from pathlib import Path

print(f"Python {sys.version.split()[0]}")

# PY is the interpreter the later cells install into and serve from. It is Colab's own unless
# this primitive asks for another Python (WANT, empty when it does not say): then a virtual
# environment on that Python is made with uv, which fetches the interpreter itself.
PY = sys.executable
WANT = ""
if WANT and not sys.version.startswith(WANT + "."):
    VENV = "/content/venv"
    try:
        # --clear: running the notebook again in the same session finds the environment of the
        # first run, and uv refuses to make one over it.
        for step in (
            [sys.executable, "-m", "pip", "install", "-q", "uv"],
            [sys.executable, "-m", "uv", "venv", "--clear", "--seed", "--python", WANT, VENV],
        ):
            subprocess.run(step, check=True, capture_output=True, text=True)
        PY = VENV + "/bin/python"
        # Colab sets PYTHONPATH to its own packages, built for its own Python.
        os.environ.pop("PYTHONPATH", None)
        print(f"This primitive asks for Python {WANT}: using it in {VENV}")
    except subprocess.CalledProcessError as exc:
        print(f"Could not set up Python {WANT}; staying on {sys.version.split()[0]}:")
        print((exc.stderr or "")[-600:])

def declared():
    # A fresh environment has none of what Colab ships (numpy, pandas ...). A requirements.txt
    # that leans on Colab for them (ANDES) or is absent (pvlib) leaves the endpoint unable to
    # import, so pyproject's dependencies go in too. Colab's own Python needs nothing added.
    if PY == sys.executable:
        return []
    try:
        import tomllib
        table = tomllib.loads(Path(WORKDIR, "pyproject.toml").read_text())
        return list(table.get("project", {}).get("dependencies", []))
    except Exception:
        return []


# The file nearest the root first: a requirements.txt under docs/ or tests/ is not this
# primitive's, and rglob alone would pick whichever it met first.
found = sorted(
    Path(WORKDIR).rglob("requirements.txt"),
    key=lambda p: (len(p.relative_to(WORKDIR).parts), str(p)),
)
if not found:
    print("No requirements.txt. Install what you need with pip, e.g. !pip install pandas")
    if PY != sys.executable:
        done = subprocess.run(
            [PY, "-m", "pip", "install", "-q", "fastapi", "uvicorn", *declared()],
            capture_output=True, text=True,
        )
        if done.returncode != 0:
            print("pip failed:")
            print(done.stderr[-2000:])
else:
    req = found[0]
    print(f"Installing {req.relative_to(WORKDIR)} …")
    if len(found) > 1:
        others = ", ".join(str(p.relative_to(WORKDIR)) for p in found[1:4])
        print(f"Not installed (only one file is): {others}")

    def versions():
        return {d.metadata["Name"].lower(): d.version for d in md.distributions() if d.metadata["Name"]}

    def install():
        # The endpoint needs FastAPI and uvicorn, which Colab ships and a fresh environment does
        # not. Asked for in the same install, pip picks versions that fit the primitive's pins.
        serve = ["fastapi", "uvicorn"] if PY != sys.executable else []
        return subprocess.run(
            [PY, "-m", "pip", "install", "-q", "-r", str(req), *serve, *declared()],
            capture_output=True, text=True,
        )

    before = versions()
    done = install()
    if done.returncode != 0 and PY != sys.executable:
        # Keep the requested Python only if it works: fall back to Colab's, and say so.
        print(f"pip failed on Python {WANT}; trying Colab's {sys.version.split()[0]} instead.")
        PY = sys.executable
        done = install()
    if done.returncode == 0:
        print("done")
        after = versions()
        # An install can replace what Colab ships; say so, so a later surprise has a cause.
        for name in sorted(n for n in before if n in after and before[n] != after[n])[:12]:
            print(f"  changed: {name} {before[name]} -> {after[name]}")
    else:
        print(
            "pip failed. pip installs the whole file or none of it, so one package that "
            "cannot be built on this Python stops all of it:"
        )
        print(done.stderr[-2000:])


In [ ]:
import os
os.chdir(WORKDIR)

readme = next((p for p in sorted(Path(WORKDIR).rglob("README*")) if p.is_file()), None)
print(readme.read_text(errors="replace")[:3000] if readme else "No README in this primitive.")


## Your turn

`@zz-windfm-test/akkudoktor-eos` is unpacked in `/content/primitive` and it is the working directory. Add a
cell and use it.

Nothing here is sent back to the Hub — this session is yours, and it disappears when
you close it. Colab's free tier gives you CPU always and a GPU when one is spare.
